In [ ]:
# Consolidated imports required by the notebook
# These libraries are used for filesystem operations, data manipulation, numerical operations, and serialization
import os
import pandas as pd
import numpy as np
import pickle


# Tabular EDA and Preprocessing Overview
This notebook performs exploratory data preparation for tabular clinical data. It loads label and VAS sheets from the project data folder,
standardizes identifiers and visit labeling, reshapes VAS data to a long format, merges labels and scores, creates derived features,
encodes categorical fields as binary indicators, filters rows with invalid diagnostic or quality flags, and saves a checkpoint of the cleaned dataframe.

Run the cells in order after the consolidated imports cell at the top. Do not change execution order when reproducing results.

## Data loading and header flattening
This section locates the Excel workbook in the `Data` directory, reads the label sheet and the VAS sheet,
and flattens the VAS sheet's two-row header into a single-level header suitable for dataframe operations.

In [ ]:
# Build the path to the Excel workbook located in the project 'Data' directory
data_dir = os.path.join(os.getcwd(), 'Data')
file_name = 'KP02-Pain maps and VAS 28Jul2022.xlsx'
full_file_path = os.path.join(data_dir, file_name)

# Read label and VAS sheets into dataframes; any I/O error will be printed and re-raised
try:
    df_labels = pd.read_excel(full_file_path, sheet_name='Pain Map Labels', header=0)
    df_vas = pd.read_excel(full_file_path, sheet_name='VAS', header=[0, 1])
except Exception as e:
    print(f"Error loading Excel file: {e}")
    raise

# Convert the VAS sheet's two-level column index into a single-level column index
new_columns = []
for col in df_vas.columns:
    top_level = str(col[0])
    bottom_level = str(col[1])
    if top_level.startswith("Unnamed"):
        new_columns.append(bottom_level)
    else:
        new_columns.append(f"{top_level}_{bottom_level}")

df_vas.columns = new_columns

print("Data loaded and headers flattened!")

## Standardize patient identifiers and column names
This section normalizes the patient identifier field in both VAS and labels tables,
handles Hebrew column naming if present, and ensures consistent formatting for downstream joins.

In [ ]:
# Locate Hebrew 'PN' header if present and rename it to 'PN' for consistency
for col in df_vas.columns:
    if 'מתנדב' in str(col):
        df_vas.rename(columns={col: 'PN'}, inplace=True)
        break

# Ensure the labels sheet uses 'Visit' as the visit column name
if 'VISIT' in df_labels.columns:
    df_labels.rename(columns={'VISIT': 'Visit'}, inplace=True)

# Normalize PN values in both dataframes: trim, uppercase, and pad single-digit suffixes
df_vas['PN'] = df_vas['PN'].astype(str).str.strip().str.upper()
df_vas['PN'] = df_vas['PN'].str.replace(r'-(\d)$', r'-0\1', regex=True)

df_labels['PN'] = df_labels['PN'].astype(str).str.strip().str.upper()
df_labels['PN'] = df_labels['PN'].str.replace(r'-(\d)$', r'-0\1', regex=True)

print("Patient IDs and column names standardized! 'PN' is ready to use.")

## Reshape VAS scores to long format
The original VAS sheet contains separate columns per visit stage. Here we extract visit-specific columns,
rename them to a common schema, and concatenate the results into a long dataframe keyed by `PN` and `Visit`.

In [ ]:
# Define mapping of visit numeric codes to the VAS column prefixes
visit_mapping = {
    1: 'Baseline',
    2: 'End of treatment',
    3: 'End of trial'
}

# Build a list of long-format pieces, one per visit, then concatenate
vas_long_pieces = []
for visit_num, prefix in visit_mapping.items():
    stage_cols = [col for col in df_vas.columns if col.startswith(prefix)]
    cols_to_keep = ['PN'] + stage_cols

    df_subset = df_vas[cols_to_keep].copy()
    df_subset.columns = ['PN'] + [col.replace(f"{prefix}_", "") for col in stage_cols]

    df_subset['Visit'] = visit_num
    vas_long_pieces.append(df_subset)

df_vas_long = pd.concat(vas_long_pieces, ignore_index=True)

print("VAS data successfully reshaped! 'df_vas_long' is now ready.")

## Merge labels with VAS scores and consolidate laterality
This section performs a left join to combine label metadata with VAS scores and then consolidates
separate left/right measurements into unified `Worst` and `Average` columns using the `SIDE_` indicator.

In [ ]:
# Merge label information with the long-format VAS scores; preserve all rows from df_labels
df_merged = pd.merge(df_labels, df_vas_long, on=['PN', 'Visit'], how='left')

# Create boolean masks for side interpretation based on the textual SIDE_ column
is_right = df_merged['SIDE_'].astype(str).str.strip().str.title() == 'Right'
is_left = df_merged['SIDE_'].astype(str).str.strip().str.title() == 'Left'

# For rows labeled 'Right', copy right-specific measurements into unified columns
df_merged.loc[is_right, 'Worst'] = df_merged.loc[is_right, 'R-Worst']
df_merged.loc[is_right, 'Average'] = df_merged.loc[is_right, 'R-Average']

# For rows labeled 'Left', copy left-specific measurements into unified columns
df_merged.loc[is_left, 'Worst'] = df_merged.loc[is_left, 'L-Worst']
df_merged.loc[is_left, 'Average'] = df_merged.loc[is_left, 'L-Average']

# Remove the original side-specific columns when present
cols_to_drop = ['R-Worst', 'L-Worst', 'R-Average', 'L-Average']
df_merged.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print("Left merge and side-consolidation complete!")

## Encode laterality as a binary feature
Convert the textual `SIDE_` indicator into a numeric binary column `side_right` where 1 denotes Right and 0 denotes Left.
The original textual column is dropped afterwards to avoid duplication.

In [ ]:
# If the SIDE_ column exists, map textual values to binary and drop the original column
if 'SIDE_' in df_merged.columns:
    df_merged['side_right'] = df_merged['SIDE_'].astype(str).str.strip().str.title().map({
        'Right': 1,
        'Left': 0
    })

    df_merged['side_right'] = df_merged['side_right'].fillna(0).astype(int)
    df_merged.drop(columns=['SIDE_'], inplace=True)
    print("Success: 'SIDE_' text removed. Created 'side_right' binary column (1=Right, 0=Left).")
else:
    print("Check: 'SIDE_' column not found. It may have already been processed.")

# Show a small sample to verify the transformation
display(df_merged[['PN', 'side_right', 'Worst', 'Average']].head(10))

## Encode sex as a binary indicator
Map existing numeric sex codes into a `male` binary column. This preserves numerical semantics used in downstream models.

In [ ]:
# Convert the 'SEX' code into a binary 'male' column and remove the original column
if 'SEX' in df_merged.columns:
    df_merged['male'] = df_merged['SEX'].map({1: 0, 2: 1})
    df_merged['male'] = df_merged['male'].fillna(0).astype(int)
    df_merged.drop(columns=['SEX'], inplace=True)
    print("Success: Converted 'SEX' (1/2) into 'male' (1/0).")
    display(df_merged[['PN', 'male']].head(10))
else:
    print("Check: 'SEX' column not found. It may have already been renamed or dropped.")

## Rename anterior columns for downstream image processing
Standardize column names that reference anterior regions to a shorter consistent naming convention used later in the pipeline.

In [ ]:
# Apply a deterministic rename mapping for anterior columns to concise names
df_merged = df_merged.rename(columns={
    "Anterior_3": "Ant_3",
    "Anterior_A5": "Ant_5",
    "Anterior_A6": "Ant_6",
    "Anterior_A7": "Ant_7",
    "Anterior_A9": "Ant_9"
})
print("Anterior columns renamed successfully.")

## Unify lateral and medial indicators
Merge split indicator columns that represent the same anatomical concept into single boolean/integer flags.

In [ ]:
# Combine lateral indicator variants using bitwise OR to preserve any positive signal
if 'Lateral_LJL' in df_merged.columns and 'Lateral_LCL' in df_merged.columns:
    df_merged['Lateral'] = (
        df_merged['Lateral_LJL'].fillna(0).astype(int) |
        df_merged['Lateral_LCL'].fillna(0).astype(int)
    )

# Combine medial indicator variants similarly
if 'Medial_MJL' in df_merged.columns and 'Medial_MCL' in df_merged.columns:
    df_merged['Medial'] = (
        df_merged['Medial_MJL'].fillna(0).astype(int) |
        df_merged['Medial_MCL'].fillna(0).astype(int)
    )

# Remove the now-redundant split columns if present
df_merged.drop(
    columns=['Lateral_LJL', 'Lateral_LCL', 'Medial_MJL', 'Medial_MCL'],
    inplace=True,
    errors='ignore'
)

print("Created unified 'Lateral' and 'Medial' columns and removed the original split columns.")
display(df_merged.head())

## Create target label 'pain_label'
Derive the modelling target from the `AKP` column. This cell enforces the requirement that `AKP` exists and converts it to numeric.

In [ ]:
# Ensure the chosen target column 'AKP' is present and create a numeric 'pain_label'
if 'AKP' not in df_merged.columns:
    raise KeyError("df_merged must contain 'AKP' to create pain_label.")

df_merged['pain_label'] = pd.to_numeric(df_merged['AKP'], errors='coerce')

print("Created 'pain_label' from 'AKP'.")
print("Value counts (including NaN):")
print(df_merged['pain_label'].value_counts(dropna=False).sort_index())

display(df_merged[['PN', 'Visit', 'AKP', 'pain_label']].head(10))

## Filter rows with invalid diagnosis or quality flags
Remove rows where `PF_DIAGNOSIS` or `QUALITY` are flagged as zero. This ensures downstream analysis only uses valid, quality-controlled records.

In [ ]:
# Validate that df_merged exists and contains required filter columns
if 'df_merged' not in globals() or not isinstance(df_merged, pd.DataFrame):
    raise NameError("df_merged is not available. Run the preprocessing cells first.")

required_cols = ['PF_DIAGNOSIS', 'QUALITY']
missing_cols = [c for c in required_cols if c not in df_merged.columns]
if missing_cols:
    raise KeyError(f"df_merged is missing required columns: {missing_cols}")

before_rows = len(df_merged)

pf_diag_num = pd.to_numeric(df_merged['PF_DIAGNOSIS'], errors='coerce')
quality_num = pd.to_numeric(df_merged['QUALITY'], errors='coerce')

remove_mask = pf_diag_num.eq(0) | quality_num.eq(0)
df_merged = df_merged.loc[~remove_mask].reset_index(drop=True)

removed_rows = int(remove_mask.sum())
after_rows = len(df_merged)

print('=== Filtering Summary ===')
print(f"Removed rows (PF_DIAGNOSIS == 0 OR QUALITY == 0): {removed_rows}")
print(f"Rows before: {before_rows}")
print(f"Rows after: {after_rows}")

## Create derived anterior pattern features
Ensure anterior indicator columns are numeric and derive `Pat` and `Ext` features based on specified anatomical rules.

In [ ]:
# Ensure specific anterior columns are numeric and default missing values to 0
ant_cols = ["Ant_3", "Ant_5", "Ant_6", "Ant_7", "Ant_9"]
for col in ant_cols:
    if col in df_merged.columns:
        df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce").fillna(0).astype(int)

# Derive 'Pat' when at least two of Ant_5, Ant_6, Ant_7 are positive
if all(col in df_merged.columns for col in ["Ant_5", "Ant_6", "Ant_7"]):
    df_merged["Pat"] = (
        df_merged[["Ant_5", "Ant_6", "Ant_7"]].sum(axis=1) >= 2
    ).astype(int)

# Derive 'Ext' when at least two of Ant_3, Ant_6, Ant_9 are positive
if all(col in df_merged.columns for col in ["Ant_3", "Ant_6", "Ant_9"]):
    df_merged["Ext"] = (
        df_merged[["Ant_3", "Ant_6", "Ant_9"]].sum(axis=1) >= 2
    ).astype(int)

print("Pat and Ext columns were added to df_merged.")
display(df_merged[["PN", "Visit", "side_right", "Ant_3", "Ant_5", "Ant_6", "Ant_7", "Ant_9", "Pat", "Ext"]].head())

## Reorder columns for improved readability
Place `Lateral`, `Medial`, `Pat`, and `Ext` immediately after `Ant_9` so that related features appear together.

In [ ]:
# Reorder dataframe columns so that prioritised features follow Ant_9
priority_cols = ["Lateral", "Medial", "Pat", "Ext"]

if "Ant_9" in df_merged.columns:
    cols = list(df_merged.columns)
    cols = [col for col in cols if col not in priority_cols]
    ant9_index = cols.index("Ant_9")
    new_cols = cols[:ant9_index + 1] + priority_cols + cols[ant9_index + 1:]
    new_cols = [col for col in new_cols if col in df_merged.columns]
    df_merged = df_merged[new_cols]

print("Columns reordered successfully.")
display(df_merged.head())

## Save a checkpoint of the cleaned dataframe
Serialize the cleaned and processed dataframe to disk so subsequent analysis can load this preprocessed state quickly.

In [ ]:
# Export the cleaned dataframe to a pickle file for later reuse
with open('cleaned_tabular_checkpoint.pkl', 'wb') as f:
    pickle.dump(df_merged, f)
print("DataFrame exported to cleaned_tabular_checkpoint.pkl")